# Module 2: Tracing

> ~25 min

Every call your agent makes is already traced to its **development** environment. In this module you generate a batch of realistic conversations, turn on **Engine** so it starts scanning them, and then explore the same traces yourself: one thread in the **Trajectory** view, the search bar, AI search, and queries from the SDK, the CLI and your coding agent. By the end you will have found, by hand, a failure pattern that Engine reports on in Module 3.

## Setup

This cell builds the agent and looks up the tracing project behind the agent's `development` environment. The SDK and CLI queries below use that project's ID.

> **Project fallback:** on self-hosted or older orgs without agents, traces go to `LANGSMITH_PROJECT` (default `langsmith-poc`), and this cell resolves that project instead.

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from collections import Counter
from datetime import datetime, timedelta, timezone

from langsmith import Client, uuid7

from agents.client_research_agent import build_client_research_agent
from utils.config import AGENT_ENVIRONMENT, AGENT_NAME, TRACING_MODE, environment_project
from utils.notebook import trace_link
from utils.traffic import generate_traffic

client = Client()
agent = build_client_research_agent()
dest = environment_project(client)
project_id = dest.project_id
since = datetime.now(timezone.utc) - timedelta(days=7)  # SDK queries default to the last 24h

print("Agent:", AGENT_NAME, "/", AGENT_ENVIRONMENT, f"({TRACING_MODE} mode)")
trace_link(dest.url, "Tracing")

## 1. Generate traffic

`generate_traffic()` replays 24 scripted conversations through the agent, five at a time, and tags each turn with `source: tour-seed`. Some threads have one turn and some have several; the mix covers client lookups, market research and a few out-of-scope asks. The run takes about 4 minutes.

In [ ]:
thread_ids = generate_traffic()

print("Threads generated:", len(thread_ids))
trace_link(dest.url, "Threads")

Open the link, then the **Threads** tab. Each row is one conversation, and the turn count shows which ones went past a single question.

<img src="../images/warm_up_traces.png" alt="Tracing table with the Threads, Traces and Runs tabs" style="width: auto; max-height: 360px; border-radius: 8px;">

## 2. Turn on Engine

**Engine** reads an environment's traces on a schedule, groups recurring failures into **issues**, and proposes fixes. It needs traces from the last 7 days before it can start, which is why traffic came first.

> **Requires:** Engine turned on for your organization: an org admin clicks **Enable Engine for this Organization** on the **Engine** page, and anyone else sees **Request access from an admin**. **Without it:** skip this section; Module 3 opens with a fallback.

1. Open **Engine** in the sidebar. Under **Choose an environment to analyze**, pick **Development** for `client-research-agent`. If Engine already analyzes another project, open the project selector and choose **Set up another project** first.
2. Optional: under **Connect your agent's code repository**, pick your fork of this repo. Leave **Open a pull request for every fix** off.
3. Under **What matters most to you?**, select **Tool Call Failures** and **Hallucinations**. Click **Custom Category**, paste the priority printed by the next cell, and click **Add**.
4. Leave **Analysis level** and **Focus on specific traces** as they are, and click **Start Analyzing**.

> **Requires:** a fork of this repo with the LangSmith GitHub App installed. **Without it:** skip step 2; Engine still files issues from traces, and Module 3 skips the pull request step.

<img src="../images/engine_config.png" alt="Engine setup panel, Find and fix your agent's issues" style="width: auto; max-height: 360px; border-radius: 8px;">

In [ ]:
ENGINE_PRIORITY = (
    "Client lookups that fail repeatedly: get_client_profile is called several "
    "times in one conversation and never returns a profile."
)

print("Custom priority:", ENGINE_PRIORITY)
trace_link(dest.url, "Tracing (Engine is in the sidebar)")

The first scan takes about 20 minutes. Keep going: the rest of this module covers that wait, and Module 3 opens with what Engine found.

## 3. A multi-turn thread

A **thread** is a series of traces that share one `thread_id`. Passing the same `thread_id` in the run config to two `invoke` calls makes them two turns of one conversation. The cell takes about a minute.

In [ ]:
TURNS = [
    "Pull up Acme Pension Fund and summarize their top holdings.",
    "Also give me a one-paragraph update on AAPL. Search at most once.",
]

thread_id = str(uuid7())
config = {"configurable": {"thread_id": thread_id}}

for question in TURNS:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config=config)
    print("Answer:", result["messages"][-1].text[:200], "\n")

print("Thread ID:", thread_id)

In **Threads**, open the row with this thread ID and switch the side panel to **Turns**. Both questions appear as cards in one conversation.

## 4. Read the trajectory

The **Trajectory** view shows a thread as the conversation the model saw: each turn's answer, the tool calls it made with their results, and subagent work inline.

In the same thread, switch to **Trajectory** (press `T`). Expand the grouped `get_client_profile` row, then click into the `research-agent` block to see the subagent's own messages.

Count the `get_client_profile` calls in the first turn. The agent passes the client's name, gets an error back, and tries again with a different value before it answers. Click one failed call to open it in **Details** (press `D`) with its error and inputs.

<img src="../images/TODO-02-trajectory-failed-lookup.png" alt="Trajectory view with repeated failed get_client_profile calls, one open in Details" style="width: auto; max-height: 360px; border-radius: 8px;">